# 01 · Ingest and validate

**Question:** is the raw file complete and consistent enough to build on, and which rows
form the modelling population?

**In:** `data/raw/US_Accidents_March23.csv` (Kaggle *US Accidents (2016–2023)*, Moosavi et
al., CC BY-NC-SA 4.0).
**Out:** `data/processed/accidents_raw.parquet` and a row-count ledger from raw records to
the target population.

All logic lives in `src/accidents/` (tested); this notebook runs it and shows the evidence.

In [1]:
import time

import numpy as np
import pandas as pd
from IPython.display import display

from accidents import clean, config, ingest

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 50)

## 1 · Read the raw file

pyarrow parses timestamps and booleans directly, so no hand-written parsing is needed.

In [2]:
started = time.perf_counter()
raw = ingest.read_raw_csv()
print(f"{len(raw):,} rows × {raw.shape[1]} columns in {time.perf_counter() - started:.1f} s")
print(f"memory: {raw.memory_usage(deep=True).sum() / 1e9:.1f} GB")

7,728,394 rows × 46 columns in 8.9 s


memory: 3.4 GB


## 2 · Schema contract

The contract checks only what the rest of the pipeline relies on: required columns,
parsed timestamps, unique IDs, Severity in 1–4, coordinates inside the contiguous US.
Out-of-range weather readings are a data-quality matter, handled in section 4.

In [3]:
problems = ingest.schema_problems(raw)
print("contract problems:", problems or "none")
assert not problems

contract problems: none


## 3 · Data dictionary

One row per column: type, missing share, distinct values and an example. High-cardinality
text (ID, Description, Street) is not a model feature; see notebook 03.

In [4]:
dictionary = pd.DataFrame(
    {
        "dtype": raw.dtypes.astype(str),
        "missing_%": (raw.isna().mean() * 100).round(2),
        "distinct": raw.nunique(dropna=True),
        "example": raw.iloc[123_456],
    }
)
display(dictionary)

,dtype,missing_%,distinct,example
ID,str,0.00,7728394,A-123463
Source,str,0.00,3,Source2
Severity,int64,0.00,4,3
Start_Time,datetime64[ns],0.00,5801064,2017-01-22 07:36:38
End_Time,datetime64[ns],0.00,6463024,2017-01-22 08:06:12
Start_Lat,float64,0.00,2449172,26.461716
Start_Lng,float64,0.00,2493905,-80.088387
End_Lat,float64,44.03,1568232,NaN
End_Lng,float64,44.03,1605835,NaN
Distance(mi),float64,0.00,22382,0.01


## 4 · Physically impossible readings

Weather readings outside physical bounds are sensor or unit errors. They are counted
here and set to missing in `clean.mask_impossible_readings`; the model then treats them
like any other missing reading.

In [5]:
rows = []
for column, (low, high) in clean.PHYSICAL_BOUNDS.items():
    values = raw[column]
    outside = ~values.between(low, high) & values.notna()
    rows.append(
        {
            "column": column,
            "bounds": f"[{low:g}, {high:g}]",
            "min": values.min(),
            "max": values.max(),
            "outside_bounds": int(outside.sum()),
            "outside_%": round(100 * outside.mean(), 4),
            "missing_%": round(100 * values.isna().mean(), 2),
        }
    )
display(pd.DataFrame(rows).set_index("column"))

,bounds,min,max,outside_bounds,outside_%,missing_%
column,,,,,,
Temperature(F),"[-60, 130]",-89.0,207.00,65,0.0008,2.12
Humidity(%),"[0, 100]",1.0,100.00,0,0.0000,2.25
Pressure(in),"[15, 32]",0.0,58.63,123,0.0016,1.82
Visibility(mi),"[0, 50]",0.0,140.00,1029,0.0133,2.29
Wind_Speed(mph),"[0, 150]",0.0,1087.00,51,0.0007,7.39
Precipitation(in),"[0, 12]",0.0,36.47,7,0.0001,28.51


## 5 · Store as Parquet

Parquet keeps types and loads about ten times faster than the CSV, so later notebooks
start from it.

In [6]:
ingest.write_parquet(raw)
size_csv = config.RAW_CSV.stat().st_size / 1e9
size_parquet = config.RAW_PARQUET.stat().st_size / 1e9
print(f"CSV {size_csv:.2f} GB → Parquet {size_parquet:.2f} GB")
check = pd.read_parquet(config.RAW_PARQUET, columns=["ID"])
assert len(check) == len(raw)

CSV 3.06 GB → Parquet 0.79 GB


## 6 · Row-count ledger: raw records → target population

Each step of decision D4 and the rows it removes. The population is Source1 records
from its stable labelling regime (from 2020-10-01), Severity 2 or 4; notebook 02 shows
why.

In [7]:
source1 = raw["Source"] == config.SOURCE
in_regime = (raw["Start_Time"] >= config.REGIME_START) & (raw["Start_Time"] < config.DATA_END)
labelled = raw["Severity"].isin([2, 4])
steps = [
    ("all raw records", np.ones(len(raw), dtype=bool)),
    ("Source1 only", source1),
    ("in stable regime (2020-10-01 → 2023-03-31)", source1 & in_regime),
    ("Severity 2 or 4", source1 & in_regime & labelled),
]
ledger = pd.DataFrame([{"step": name, "rows": int(mask.sum())} for name, mask in steps]).set_index(
    "step"
)
ledger["removed"] = (-ledger["rows"].diff()).fillna(0).astype(int)
display(ledger)

population = clean.target_window(raw)
print(f"target population: {len(population):,} rows, Severity-4 share {population['y'].mean():.4f}")
print(
    "severity values removed in step 4:",
    raw.loc[source1 & in_regime & ~labelled, "Severity"].value_counts().to_dict(),
)

,rows,removed
step,,
all raw records,7728394,0
Source1 only,4325632,3402762
in stable regime (2020-10-01 → 2023-03-31),3242955,1082677
Severity 2 or 4,3242942,13


target population: 3,242,942 rows, Severity-4 share 0.0256
severity values removed in step 4: {3: 10, 1: 3}


## Decisions and evidence

- The raw file (7,728,394 records, 46 columns) meets the schema contract: every required
  column present, all timestamps parsed, IDs unique, Severity within 1–4, coordinates
  inside the contiguous US.
- Impossible weather readings exist but are rare: at most 0.013% of a column (visibility
  above 50 mi), with values such as 207 °F, 1,087 mph and 0 inHg. They become missing values
  rather than being dropped, so no accident record is lost.
- Parquet cuts the file from 3.06 GB to 0.79 GB and keeps every row.
- The modelling population is 3,242,942 records with a Severity-4 share of 2.56%. Step 4
  of the ledger drops just 13 stray Severity 1 and 3 records from a regime that otherwise
  uses only 2 and 4.
- Next: notebook 02 shows why the population is defined this way, and maps missingness,
  space, time, weather and road context.